# 02. RF-DETR 영상 inference 시각화

이 노트북은 RF-DETR checkpoint로 영상 파일 또는 영상 폴더를 inference하고, bbox와 `class score` 라벨이 그려진 MP4 영상을 저장합니다.

RF-DETR 환경은 YOLO11 환경과 다르므로 `ltb_rfdetr` kernel에서 실행하는 것을 권장합니다.

기본값은 안전하게 실행되지 않도록 되어 있습니다.

- `RUN_PREVIEW = False`: 영상 목록과 metadata만 확인하는 preview도 기본 비활성화
- `RUN_INFERENCE = False`: 실제 model load/inference/video write 비활성화
- `DRY_RUN = True`: inference cell을 실행해도 preview만 수행

RF-DETR은 frame을 임시 jpg로 저장한 뒤 `model.predict(image_path, threshold=CONF)` 방식으로 처리합니다. 속도보다 호환성과 안정성을 우선한 구현입니다.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").is_file() and (path / "src").is_dir():
            return path
    raise RuntimeError("repo root를 찾지 못했습니다. notebook을 repo 안에서 실행하거나 REPO_ROOT를 직접 지정하세요.")

REPO_ROOT = find_repo_root(Path.cwd())
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("REPO_ROOT:", REPO_ROOT)

from labelstudio_bbox_tools.video_inference.classes import load_class_names, make_class_color_map
from labelstudio_bbox_tools.video_inference.rfdetr import run_rfdetr_video_inference


In [ ]:
# ===== 사용자가 주로 바꾸는 설정 =====

INPUT_PATH = Path("/path/to/video_or_video_folder")
OUT_DIR = Path("/path/to/video_inference_outputs")
MODEL_WEIGHTS = Path("/path/to/checkpoint_best_total.pth")
CLASS_YAML = Path("/path/to/data.yaml")
MANUAL_CLASSES = []  # CLASS_YAML을 쓰면 비워둡니다. yaml 없이 직접 넣을 때만 사용하세요.

MODEL_VARIANT = "medium"  # "auto", "nano", "small", "medium", "large"
DEVICE = "cuda"           # RF-DETR 환경에 맞게 "cuda", "cpu", None 등을 사용합니다.
CONF = 0.25
IOU = 0.45                # RF-DETR 후처리와 별개로 classwise NMS에 한 번 더 사용됩니다.

RECURSIVE = False
MAX_VIDEOS = 1
FRAME_STRIDE = 1
START_SECONDS = None
END_SECONDS = None
MAX_FRAMES = 300

EXPECTED_CLASS_COUNT = 28
STRICT_CLASS_COUNT = True

FONT_PATH = None
FONT_SIZE = 20
LINE_WIDTH = 3
RUN_NAME = None
OVERWRITE = False
TEMP_FRAME_DIR = None     # None이면 실행 중 임시 폴더를 만들고 종료 시 삭제합니다.

RUN_PREVIEW = False
RUN_INFERENCE = False
DRY_RUN = True


In [ ]:
# class YAML과 color map 확인 cell입니다.

if CLASS_YAML.exists() or MANUAL_CLASSES:
    class_names = load_class_names(
        class_yaml=CLASS_YAML if CLASS_YAML.exists() else None,
        manual_classes=MANUAL_CLASSES or None,
        expected_count=EXPECTED_CLASS_COUNT,
        strict_count=STRICT_CLASS_COUNT,
    )
    color_map = make_class_color_map(class_names)
    print("class_count:", len(class_names))
    for idx, name in enumerate(class_names):
        print(f"{idx:02d}: {name:24s} color={color_map[name]}")
else:
    print("CLASS_YAML 경로를 실제 data.yaml로 수정하거나 MANUAL_CLASSES를 채워주세요.")


In [ ]:
# 영상 목록과 metadata만 확인합니다. model은 load하지 않습니다.

if not RUN_PREVIEW:
    print("RUN_PREVIEW=False 이므로 preview를 실행하지 않습니다.")
else:
    preview = run_rfdetr_video_inference(
        input_path=INPUT_PATH,
        out_dir=OUT_DIR,
        model_weights=MODEL_WEIGHTS,
        class_yaml=CLASS_YAML if CLASS_YAML.exists() else None,
        manual_classes=MANUAL_CLASSES or None,
        model_variant=MODEL_VARIANT,
        recursive=RECURSIVE,
        max_videos=MAX_VIDEOS,
        expected_class_count=EXPECTED_CLASS_COUNT,
        strict_class_count=STRICT_CLASS_COUNT,
        run_name=RUN_NAME,
        dry_run=True,
    )
    print(preview.as_dict())


In [ ]:
# 실제 RF-DETR inference와 시각화 영상 저장 cell입니다.

if not RUN_INFERENCE:
    print("RUN_INFERENCE=False 이므로 실제 inference를 실행하지 않습니다.")
else:
    result = run_rfdetr_video_inference(
        input_path=INPUT_PATH,
        out_dir=OUT_DIR,
        model_weights=MODEL_WEIGHTS,
        class_yaml=CLASS_YAML if CLASS_YAML.exists() else None,
        manual_classes=MANUAL_CLASSES or None,
        model_variant=MODEL_VARIANT,
        device=DEVICE,
        conf=CONF,
        iou=IOU,
        recursive=RECURSIVE,
        max_videos=MAX_VIDEOS,
        frame_stride=FRAME_STRIDE,
        start_seconds=START_SECONDS,
        end_seconds=END_SECONDS,
        max_frames=MAX_FRAMES,
        expected_class_count=EXPECTED_CLASS_COUNT,
        strict_class_count=STRICT_CLASS_COUNT,
        run_name=RUN_NAME,
        font_path=FONT_PATH,
        font_size=FONT_SIZE,
        line_width=LINE_WIDTH,
        overwrite=OVERWRITE,
        temp_frame_dir=TEMP_FRAME_DIR,
        dry_run=DRY_RUN,
    )
    print(result.as_dict())
